In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.jfk_july_test
USING DELTA
AS
SELECT *
FROM flight_delay_predictor_project.default.jfk_july2026_flight_data
WHERE ORIGIN = 'JFK';

In [0]:
import requests
import pandas as pd

resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 40.6413,
        "longitude": -73.7781,
        "start_date": "2026-07-01",
        "end_date": "2026-07-31",
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,snowfall,"
                  "wind_speed_10m,wind_direction_10m,wind_gusts_10m,"
                  "cloud_cover,pressure_msl,weather_code",
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
        "precipitation_unit": "inch",
        "timezone": "America/New_York",
    },
    timeout=60,
)

resp.raise_for_status()

pdf = pd.DataFrame(resp.json()["hourly"])
pdf["time"] = pd.to_datetime(pdf["time"])

df = spark.createDataFrame(pdf)

df.write.mode("overwrite").saveAsTable(
    "flight_delay_predictor_project.default.jfk_july_weather"
)

display(df)

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.july_flight_and_weather_table AS

WITH flights_parsed AS (
    SELECT
        *,
        COALESCE(
            try_to_timestamp(CAST(FL_DATE AS STRING), 'M/d/yyyy h:mm:ss a'),
            try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd HH:mm:ss'),
            try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd')
        ) AS fl_date_ts,
        CAST(CRS_DEP_TIME AS INT) AS crs_dep_int
    FROM flight_delay_predictor_project.default.jfk_july_test
),

flights_hourly AS (
    SELECT
        *,
        timestampadd(
            HOUR,
            crs_dep_int DIV 100,
            fl_date_ts
        ) AS dep_hour,
        timestampadd(
            MINUTE,
            crs_dep_int % 100,
            timestampadd(HOUR, crs_dep_int DIV 100, fl_date_ts)
        ) AS crs_dep_ts
    FROM flights_parsed
)

SELECT
    f.*,
    w.* EXCEPT (time)
FROM flights_hourly f
LEFT JOIN flight_delay_predictor_project.default.jfk_july_weather w
    ON f.dep_hour = w.time;

In [0]:
%sql
SELECT
    COUNT(*) AS total_flights,
    COUNT(temperature_2m) AS flights_with_weather,
    COUNT(*) - COUNT(temperature_2m) AS flights_missing_weather
FROM flight_delay_predictor_project.default.july_flight_and_weather_table;

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.jfk_july_final_table AS

WITH flights_parsed AS (
    SELECT
        *,
        COALESCE(
            try_to_timestamp(CAST(FL_DATE AS STRING), 'M/d/yyyy h:mm:ss a'),
            try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd HH:mm:ss'),
            try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd')
        ) AS fl_date_ts,
        CAST(CRS_DEP_TIME AS INT) AS crs_dep_int
    FROM flight_delay_predictor_project.default.jfk_july_test
),

flights_hourly AS (
    SELECT
        *,
        timestampadd(
            HOUR,
            crs_dep_int DIV 100,
            fl_date_ts
        ) AS dep_hour
    FROM flights_parsed
)

SELECT
    f.*,

    -- Departure-hour weather
    w0.temperature_2m,
    w0.relative_humidity_2m,
    w0.precipitation,
    w0.snowfall,
    w0.wind_speed_10m,
    w0.wind_direction_10m,
    w0.wind_gusts_10m,
    w0.cloud_cover,
    w0.pressure_msl,
    w0.weather_code,

    -- Weather 1 hour after departure
    w1.temperature_2m AS temperature_2m_plus1h,
    w1.relative_humidity_2m AS relative_humidity_2m_plus1h,
    w1.precipitation AS precipitation_plus1h,
    w1.snowfall AS snowfall_plus1h,
    w1.wind_speed_10m AS wind_speed_10m_plus1h,
    w1.wind_direction_10m AS wind_direction_10m_plus1h,
    w1.wind_gusts_10m AS wind_gusts_10m_plus1h,
    w1.cloud_cover AS cloud_cover_plus1h,
    w1.pressure_msl AS pressure_msl_plus1h,
    w1.weather_code AS weather_code_plus1h,

    -- Weather 2 hours after departure
    w2.temperature_2m AS temperature_2m_plus2h,
    w2.relative_humidity_2m AS relative_humidity_2m_plus2h,
    w2.precipitation AS precipitation_plus2h,
    w2.snowfall AS snowfall_plus2h,
    w2.wind_speed_10m AS wind_speed_10m_plus2h,
    w2.wind_direction_10m AS wind_direction_10m_plus2h,
    w2.wind_gusts_10m AS wind_gusts_10m_plus2h,
    w2.cloud_cover AS cloud_cover_plus2h,
    w2.pressure_msl AS pressure_msl_plus2h,
    w2.weather_code AS weather_code_plus2h


FROM flights_hourly f

LEFT JOIN flight_delay_predictor_project.default.jfk_july_weather w0
    ON f.dep_hour = w0.time

LEFT JOIN flight_delay_predictor_project.default.jfk_july_weather w1
    ON timestampadd(HOUR, 1, f.dep_hour) = w1.time

LEFT JOIN flight_delay_predictor_project.default.jfk_july_weather w2
    ON timestampadd(HOUR, 2, f.dep_hour) = w2.time

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.jfk_july_test_table AS
SELECT
  YEAR, MONTH, DAY_OF_MONTH, DAY_OF_WEEK, FL_DATE, ORIGIN, ORIGIN_CITY_NAME, DEST, DEST_CITY_NAME,
  CRS_DEP_TIME, DEP_TIME, DEP_DELAY, DEP_DELAY_NEW, DEP_DEL15, CRS_ARR_TIME, ARR_DELAY,
  CANCELLED, CANCELLATION_CODE, WEATHER_DELAY,
  'August' AS source_month,
  fl_date_ts, crs_dep_int, dep_hour,
  to_timestamp(
    concat(date_format(fl_date_ts, 'yyyy-MM-dd'), ' ', lpad(CAST(crs_dep_int AS STRING), 4, '0')),
    'yyyy-MM-dd HHmm'
  ) AS crs_dep_ts,
  temperature_2m, relative_humidity_2m, precipitation, snowfall, wind_speed_10m, wind_direction_10m,
  wind_gusts_10m, cloud_cover, pressure_msl, weather_code,
  temperature_2m_plus1h, precipitation_plus1h, snowfall_plus1h, wind_speed_10m_plus1h,
  wind_gusts_10m_plus1h, cloud_cover_plus1h, weather_code_plus1h, pressure_msl_plus1h,
  temperature_2m_plus2h, precipitation_plus2h, snowfall_plus2h, wind_speed_10m_plus2h,
  wind_gusts_10m_plus2h, cloud_cover_plus2h, weather_code_plus2h, pressure_msl_plus2h,
  wind_direction_10m_plus1h, relative_humidity_2m_plus2h

FROM flight_delay_predictor_project.default.jfk_july_final_table;

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.jfk_july_test_table AS
SELECT
  temperature_2m, relative_humidity_2m, precipitation, snowfall, wind_speed_10m, wind_direction_10m,
  wind_gusts_10m, cloud_cover, pressure_msl, weather_code,
  temperature_2m_plus1h, precipitation_plus1h, snowfall_plus1h, wind_speed_10m_plus1h,
  wind_gusts_10m_plus1h, cloud_cover_plus1h, weather_code_plus1h, pressure_msl_plus1h,
  temperature_2m_plus2h, precipitation_plus2h, snowfall_plus2h, wind_speed_10m_plus2h,
  wind_gusts_10m_plus2h, cloud_cover_plus2h, weather_code_plus2h, pressure_msl_plus2h,
  wind_direction_10m_plus1h, relative_humidity_2m_plus2h

FROM flight_delay_predictor_project.default.jfk_july_final_table;

In [0]:
%run "./Final prediction"

In [0]:
from pyspark.sql import functions as F
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import requests
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

july = spark.table("flight_delay_predictor_project.default.jfk_july_final_table")

july = july.fillna({"weather_delay": 0})
july = july.withColumn("delay_in_hr",
    F.when(F.col("weather_delay") < 1, "no delay")
     .when(F.col("weather_delay") < 60, "less than 1") 
     .when(F.col("weather_delay") < 120, "1+")
     .when(F.col("weather_delay") < 300, "2-4+")
     .otherwise("5+")) 

feature_cols = list(X.columns)
july_pd = july.select(feature_cols + ["delay_in_hr"]).toPandas().dropna()

# feature_cols contains duplicate names (pressure_msl and wind_direction_10m
# are each listed twice in the training features), and pandas label-based
# selection expands duplicated labels. Select by position instead so X_aug has
# exactly the columns the model was fitted with, in the same order.
july_cols = list(july_pd.columns)
X_july = july_pd.iloc[:, [july_cols.index(c) for c in feature_cols]]
y_july = july_pd["delay_in_hr"]

july_pd["predicted"] = dtree.predict(X_july)

print("August accuracy:", accuracy_score(y_july, july_pd["predicted"]))
print(classification_report(y_july, july_pd["predicted"], zero_division=0))
print(confusion_matrix(y_july, july_pd["predicted"]))

display(july_pd.groupby(["delay_in_hr", "predicted"]).size().reset_index(name="flights"))